# Módulo I4 · Variables, pipelines y gradient boosting

**Caso TiendaSur:** la tienda quiere saber, en el momento de la compra, **qué pedidos se van a devolver**, para actuar antes (verificar la talla, avisar al cliente, enviar con comprobación). Cada devolución cuesta dinero; cada acción preventiva también.

Datos simulados a nivel de **pedido** y **cliente**: ~24.000 pedidos de 1.800 clientes en dos años. Cada cliente hace varios pedidos y la tasa de devolución sube en enero y diciembre y con un lote defectuoso de máquinas desde abril de 2025. Eso hace que la **validación** importe de verdad.

Hilo del notebook:
1. Datos → 2. Variables → 3. Codificación de categorías → 4. **Fugas** (las que inflan la métrica) → 5. Validación honesta → 6. XGBoost / LightGBM / HistGradientBoosting → 7. Desequilibrio → 8. Búsqueda de hiperparámetros → 9. Interpretabilidad (permutación y SHAP) → 10. Calibración → 11. Umbral por coste → 12. Informe para negocio → ejercicios.

> **macOS:** XGBoost y LightGBM necesitan la librería OpenMP (`brew install libomp`). Si no está, el notebook no se rompe: avisa y sigue con `HistGradientBoostingClassifier`, que viene con scikit-learn. En Colab y Linux funciona sin hacer nada.

In [ ]:
import importlib
import os

os.environ.setdefault("OMP_NUM_THREADS", "4")   # evita la sobresuscripción de hilos (más lento en portátiles)
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False})

BASE = Path("proyecto-devoluciones")     # aquí se guardan figuras e informe (está en .gitignore)
BASE.mkdir(exist_ok=True)


def importar_opcional(nombre):
    """Importa una librería pesada; si falla (p. ej. falta libomp en macOS), el notebook sigue."""
    try:
        return importlib.import_module(nombre)
    except Exception as error:
        print(f"[aviso] {nombre} no disponible ({type(error).__name__}). "
              "En macOS: `brew install libomp`. Seguimos con lo que haya.")
        return None


xgb = importar_opcional("xgboost")
lgb = importar_opcional("lightgbm")
shap = importar_opcional("shap")
optuna = importar_opcional("optuna")
if optuna:
    optuna.logging.set_verbosity(optuna.logging.WARNING)

# Motor principal: el primero que esté disponible.
MOTOR = "lightgbm" if lgb else "xgboost" if xgb else "hgb"
print("Motor principal de boosting:", MOTOR)
print("XGBoost:", xgb.__version__ if xgb else "no", "| LightGBM:", lgb.__version__ if lgb else "no",
      "| SHAP:", shap.__version__ if shap else "no", "| Optuna:", optuna.__version__ if optuna else "no")

## 1 · Los datos: pedidos con historia

Fíjate en tres cosas: (1) hay **varios pedidos por cliente** (esto crea la fuga por grupo); (2) los pedidos tienen **fecha** (esto crea la fuga temporal); (3) la columna `motivo_devolucion` solo existe cuando ya ha habido devolución (esto es una fuga de objetivo esperando a que alguien la use).

In [ ]:
CATEGORIAS = ["cafe", "menaje", "maquinas", "accesorios"]
CANALES = ["web", "app", "tienda"]
PAGOS = ["tarjeta", "bizum", "contrareembolso", "paypal"]


def generar_pedidos(n_clientes: int = 1800, semilla: int = 7) -> pd.DataFrame:
    """Pedidos simulados de TiendaSur (2024-2025). Cada cliente hace varios pedidos.

    Hay una «propensión a devolver» propia de cada cliente que el modelo NO ve directamente:
    solo se nota a través de su historial. Eso es lo que hace que la fuga por cliente sea real.
    """
    rng = np.random.default_rng(semilla)
    ini, fin = pd.Timestamp("2024-01-01"), pd.Timestamp("2025-12-31")
    dias_total = (fin - ini).days
    cli = pd.DataFrame({
        "cliente_id": np.arange(n_clientes),
        "alta": ini + pd.to_timedelta(rng.integers(-700, dias_total - 60, n_clientes), unit="D"),
        "propension": rng.normal(0, 1.1, n_clientes),                 # latente
        "frecuencia": rng.gamma(2.0, 0.9, n_clientes) + 0.3,          # pedidos por trimestre
        "prefiere": rng.choice(CANALES, n_clientes, p=[.5, .3, .2]),
    })
    filas = []
    for c in cli.itertuples():
        t0 = max(c.alta, ini)
        n = max(rng.poisson(c.frecuencia * (fin - t0).days / 91), 1)
        fechas = t0 + pd.to_timedelta(np.sort(rng.integers(0, max((fin - t0).days, 1), n)), unit="D")
        filas += [(c.cliente_id, f) for f in fechas]
    p = pd.DataFrame(filas, columns=["cliente_id", "fecha_pedido"]).merge(cli, on="cliente_id")
    n = len(p)
    p["categoria"] = rng.choice(CATEGORIAS, n, p=[.45, .25, .12, .18])
    p["canal"] = np.where(rng.random(n) < .75, p["prefiere"], rng.choice(CANALES, n))
    p["metodo_pago"] = rng.choice(PAGOS, n, p=[.55, .2, .1, .15])
    precio = {"cafe": 18, "menaje": 45, "maquinas": 160, "accesorios": 25}
    p["n_articulos"] = 1 + rng.poisson(np.where(p.categoria == "cafe", 1.2, .5))
    p["importe"] = (p.categoria.map(precio) * p.n_articulos * rng.lognormal(0, .35, n)).round(2)
    p["descuento_pct"] = np.where(rng.random(n) < .3, rng.choice([10, 15, 20, 30, 40], n), 0)
    p["dias_entrega"] = np.clip(rng.poisson(np.where(p.canal == "tienda", 0.3, 3.2)), 0, 14)
    mes = p.fecha_pedido.dt.month
    z = (-3.2 + 0.9 * p.propension
         + p.categoria.map({"cafe": -0.6, "menaje": 0.2, "maquinas": 0.5, "accesorios": 0.0})
         + 0.02 * p.descuento_pct + 0.9 * ((p.n_articulos >= 3) & (p.categoria != "cafe"))
         + 1.4 * ((p.descuento_pct >= 20) & (p.canal == "web"))               # interacciones: lo que
         + 1.4 * ((p.categoria == "maquinas") & (p.dias_entrega >= 5))        # un modelo lineal no ve
         + 0.35 * (p.canal == "web") - 0.6 * (p.canal == "tienda")
         + 0.5 * (p.metodo_pago == "contrareembolso")
         + 0.2 * np.clip(p.dias_entrega - 4, 0, None)
         + 0.9 * mes.isin([1, 2]) + 0.6 * (mes == 12) + 0.4 * (mes == 11)
         + 0.0003 * np.clip(p.importe - 150, 0, None)
         + 0.7 * ((p.categoria == "maquinas") & (p.fecha_pedido >= "2025-04-01")))   # lote con defecto
    p["devuelto"] = (rng.random(n) < 1 / (1 + np.exp(-z))).astype(int)
    # Esta columna solo se rellena DESPUÉS de que el cliente devuelva: es una fuga de objetivo clásica.
    p["motivo_devolucion"] = np.where(p.devuelto == 1,
                                      rng.choice(["talla", "defecto", "no_gusta", "retraso"], n), None)
    p = p.sort_values(["fecha_pedido", "cliente_id"]).reset_index(drop=True)
    p.insert(0, "pedido_id", np.arange(len(p)))
    return p[["pedido_id", "cliente_id", "fecha_pedido", "categoria", "canal", "metodo_pago", "n_articulos",
              "importe", "descuento_pct", "dias_entrega", "devuelto", "motivo_devolucion"]]


pedidos = generar_pedidos()
print(f"{len(pedidos):,} pedidos de {pedidos.cliente_id.nunique():,} clientes "
      f"({pedidos.fecha_pedido.min().date()} a {pedidos.fecha_pedido.max().date()})")
print(f"Tasa de devolución global: {pedidos.devuelto.mean():.1%}  ->  clase positiva minoritaria")
print("Pedidos por cliente:", pedidos.groupby("cliente_id").size().describe()[["min", "50%", "max"]].to_dict())
display(pedidos.head())
display(pedidos.groupby(pedidos.fecha_pedido.dt.to_period("Q")).devuelto.agg(tasa="mean", pedidos="size").round(3).T)

## 2 · Ingeniería de variables

Cuatro familias, todas calculadas **con información disponible en el momento del pedido**:
- **Fechas** → mes y día de la semana (el modelo no entiende `2025-07-06`, sí «julio» y «domingo»).
- **Ratios** → `ahorro_eur`, `importe_por_articulo`, importe frente a lo que suele comprar el cliente.
- **Agregados por cliente** → nº de pedidos previos, días desde el anterior, tasa de devolución previa, pedidos en 90 días.
- **Categorías** → tipo `category` para que el boosting las trate de forma nativa.

El detalle que separa a un junior de un sénior: la **tasa de devolución previa** solo puede usar pedidos cuya devolución **ya se conocía**. Un pedido de hace 10 días aún puede devolverse, así que no cuenta (`VENTANA_DEV = 30`). Mira el cliente 3: `dev_resueltas` va por detrás de `n_pedidos_prev`.

In [ ]:
VENTANA_DEV = 30     # días que tarda en «cerrarse» el plazo de devolución de un pedido
PRIOR_DEV, PESO_PRIOR = 0.14, 5   # tasa de partida (conocimiento de negocio) y peso del suavizado


def historial_cliente(df: pd.DataFrame) -> pd.DataFrame:
    """Variables del historial de cada cliente calculadas SOLO con información anterior al pedido.

    Detalle clave: la etiqueta `devuelto` de un pedido no se conoce hasta ~30 días después. Por eso
    la tasa de devolución previa solo cuenta pedidos con al menos VENTANA_DEV días de antigüedad.
    """
    d = df.sort_values(["cliente_id", "fecha_pedido", "pedido_id"])
    cols = {k: np.zeros(len(d)) for k in
            ["n_pedidos_prev", "dias_desde_prev", "importe_medio_prev", "dev_resueltas", "dev_prev", "pedidos_90d"]}
    pos = 0
    for _, g in d.groupby("cliente_id", sort=False):
        dia = g.fecha_pedido.values.astype("datetime64[D]").astype(int)
        y, imp = g.devuelto.values, g.importe.values
        cy, ci = np.r_[0, np.cumsum(y)], np.r_[0, np.cumsum(imp)]
        for i in range(len(g)):
            k = min(np.searchsorted(dia, dia[i] - VENTANA_DEV, side="right"), i)   # etiquetas ya conocidas
            cols["n_pedidos_prev"][pos + i] = i
            cols["dias_desde_prev"][pos + i] = dia[i] - dia[i - 1] if i else np.nan
            cols["importe_medio_prev"][pos + i] = ci[i] / i if i else np.nan
            cols["dev_resueltas"][pos + i] = k
            cols["dev_prev"][pos + i] = cy[k]
            cols["pedidos_90d"][pos + i] = i - np.searchsorted(dia, dia[i] - 90, side="left")
        pos += len(g)
    h = pd.DataFrame(cols, index=d.index)
    h["tasa_dev_prev"] = (h.dev_prev + PRIOR_DEV * PESO_PRIOR) / (h.dev_resueltas + PESO_PRIOR)
    return h.loc[df.index]


def construir_variables(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()
    # 1) Fechas: lo que importa es el calendario, no la fecha en bruto
    d["mes"] = d.fecha_pedido.dt.month
    d["dia_semana"] = d.fecha_pedido.dt.dayofweek
    # 2) Ratios: relaciones que un árbol tardaría mucho en aprender solo
    d["ahorro_eur"] = d.importe * d.descuento_pct / 100
    d["importe_por_articulo"] = d.importe / d.n_articulos
    # 3) Agregados por cliente (solo pasado)
    d = d.join(historial_cliente(d))
    d["ratio_importe_vs_medio"] = d.importe / d.importe_medio_prev
    d["dias_desde_primer_pedido"] = (d.fecha_pedido - d.groupby("cliente_id").fecha_pedido.transform("min")).dt.days
    # 4) Categorías como tipo `category`: los tres motores de boosting las entienden de forma nativa
    for c in ["categoria", "canal", "metodo_pago"]:
        d[c] = d[c].astype("category")
    return d


t0 = time.time()
df = construir_variables(pedidos)
VARS_CAT = ["categoria", "canal", "metodo_pago"]
VARS_PEDIDO = ["mes", "dia_semana", "n_articulos", "importe", "descuento_pct", "dias_entrega",
               "ahorro_eur", "importe_por_articulo"] + VARS_CAT
VARS_HISTORIAL = ["n_pedidos_prev", "dias_desde_prev", "tasa_dev_prev", "ratio_importe_vs_medio",
                  "pedidos_90d", "dias_desde_primer_pedido"]
VARS = VARS_PEDIDO + VARS_HISTORIAL
print(f"Variables construidas en {time.time() - t0:.1f} s: {len(VARS)}")
display(df[["cliente_id", "fecha_pedido", "importe", "n_pedidos_prev", "dev_resueltas", "dev_prev", "tasa_dev_prev", "devuelto"]]
        [df.cliente_id == 3].head(8))

## 3 · Codificar categorías: one-hot, ordinal y *target encoding*

| Técnica | Cuándo | Riesgo |
|---|---|---|
| **One-hot** | pocas categorías sin orden | explosión de columnas si hay muchas |
| **Ordinal** | categorías con orden natural (bajo < medio < alto) | inventar un orden que no existe |
| **Target encoding** | muchas categorías (cliente, código postal, SKU) | **fuga** si usa la etiqueta de la propia fila |
| Nativa (`category`) | con boosting moderno | cuidado con categorías raras |

El *target encoding* sustituye cada categoría por la media de la etiqueta. Si esa media se calcula incluyendo la fila que estás codificando, **le estás dando la respuesta al modelo**. `TargetEncoder` de scikit-learn lo evita con validación cruzada interna: cada fila se codifica con un modelo que no vio su etiqueta.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, TargetEncoder

print("Cardinalidad:", {c: int(df[c].nunique()) for c in VARS_CAT + ["cliente_id"]})

# One-hot: pocas categorías sin orden (canal). Ordinal: categorías CON orden (tramo de importe).
df["tramo_importe"] = pd.cut(df.importe, [0, 30, 100, np.inf], labels=["bajo", "medio", "alto"]).astype(str)
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False).fit(df[["canal"]])
ordinal = OrdinalEncoder(categories=[["bajo", "medio", "alto"]]).fit(df[["tramo_importe"]])
print("one-hot canal ->", list(ohe.get_feature_names_out()))
print("ordinal tramo ->", dict(zip(["bajo", "medio", "alto"], ordinal.transform(pd.DataFrame({"tramo_importe": ["bajo", "medio", "alto"]})).ravel())))

# Target encoding: cliente_id tiene 1.800 valores; one-hot crearía 1.800 columnas.
# Sustituimos cada cliente por su tasa de devolución, pero HACIÉNDOLO BIEN: con validación cruzada interna.
entrenamiento_demo = df[df.fecha_pedido < "2025-07-01"]
te = TargetEncoder(cv=5, random_state=0)
con_cruce = te.fit_transform(entrenamiento_demo[["cliente_id"]], entrenamiento_demo.devuelto).ravel()   # cada fila se codifica SIN su propia etiqueta
sin_cruce = te.transform(entrenamiento_demo[["cliente_id"]]).ravel()                                      # usa el codificador final (con todas las etiquetas)
print(f"\nTarget encoding de cliente_id sobre {len(entrenamiento_demo):,} pedidos de entrenamiento")
print("  correlación con la etiqueta real, fit_transform (con cruce): "
      f"{np.corrcoef(con_cruce, entrenamiento_demo.devuelto)[0, 1]:.3f}")
print("  correlación con la etiqueta real, transform sobre el mismo set: "
      f"{np.corrcoef(sin_cruce, entrenamiento_demo.devuelto)[0, 1]:.3f}   <- más alta: ya vio la etiqueta")
print("Lección: dentro de un Pipeline, sklearn usa fit_transform (con cruce) al entrenar y transform al predecir.")

## 4 · Fugas de datos: cómo inflar una métrica sin mejorar nada

Probamos el **mismo tipo de modelo** cambiando solo qué variables entran y cómo se valida. Cuatro fugas:

1. **De objetivo:** una columna que solo existe después del evento (`motivo_devolucion`). AUC perfecto = sospecha inmediata.
2. **Temporal:** una variable que usa el futuro (la tasa de devolución del cliente calculada con todos sus pedidos).
3. **De preprocesado:** el *target encoding* calculado con todo el dataset antes de validar.
4. **Por grupo:** el mismo cliente en entrenamiento y validación; el modelo «se acuerda» de él. Se arregla con `GroupKFold`.

Compara cada fila con la **referencia honesta**. Regla práctica: si una mejora te parece demasiado buena, primero busca la fuga.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.model_selection import GroupKFold, KFold, TimeSeriesSplit, cross_val_score
from sklearn.pipeline import make_pipeline

dev = df[df.fecha_pedido < "2025-10-01"].reset_index(drop=True)      # todo lo anterior al periodo de prueba
y_dev = dev.devuelto
ALEATORIO = KFold(5, shuffle=True, random_state=0)
POR_CLIENTE = GroupKFold(5)
TEMPORAL = TimeSeriesSplit(5, gap=1000)           # gap: no mezclar pedidos cuya devolución aún no se conocía


def hgb():
    return HistGradientBoostingClassifier(max_iter=150, learning_rate=0.06, categorical_features="from_dtype", random_state=0)


def auc_cv(modelo, X, cv, grupos=None):
    return cross_val_score(modelo, X, y_dev, cv=cv, groups=grupos, scoring="roc_auc").mean()


r = {}
# Referencia honesta: variables legítimas + validación que imita el uso real (hacia delante en el tiempo)
r["Referencia honesta (temporal)"] = auc_cv(hgb(), dev[VARS], TEMPORAL)
r["Mismo modelo, KFold aleatorio"] = auc_cv(hgb(), dev[VARS], ALEATORIO)

# FUGA 1 · de objetivo: una columna que solo existe cuando ya hay devolución
dev["tiene_motivo"] = dev.motivo_devolucion.notna().astype(int)
r["Fuga 1: columna posterior al hecho"] = auc_cv(hgb(), dev[VARS + ["tiene_motivo"]], ALEATORIO)

# FUGA 2 · temporal: «tasa de devolución del cliente» calculada con TODOS sus pedidos (incluido el futuro)
dev["tasa_cliente_global"] = dev.groupby("cliente_id").devuelto.transform("mean")
r["Fuga 2: tasa de cliente con el futuro"] = auc_cv(hgb(), dev[VARS + ["tasa_cliente_global"]], TEMPORAL)

# FUGA 3 · de preprocesado: target encoding calculado en todo el dataset ANTES de validar
dev["te_cliente_ingenuo"] = dev.groupby("cliente_id").devuelto.transform("mean")      # incluye la fila de cada pedido
r["Fuga 3: target encoding fuera del CV"] = auc_cv(hgb(), dev[VARS_PEDIDO + ["te_cliente_ingenuo"]], ALEATORIO)

# Lo correcto: el encoder DENTRO del pipeline (se reajusta con cada pliegue de entrenamiento, con cruce interno)
dev["cliente"] = dev.cliente_id.astype(float)
def pipe_te():
    pre = ColumnTransformer([("te", TargetEncoder(cv=5, random_state=0), ["cliente"])], remainder="passthrough")
    return make_pipeline(pre, HistGradientBoostingClassifier(max_iter=150, learning_rate=0.06, random_state=0))
sin_cat = [c for c in VARS_PEDIDO if c not in VARS_CAT]           # esta demo usa solo las numéricas + el cliente
r["Pipeline con TE (KFold aleatorio)"] = auc_cv(pipe_te(), dev[sin_cat + ["cliente"]], ALEATORIO)

# FUGA 4 · por grupo: los mismos clientes en entrenamiento y validación
r["Pipeline con TE (GroupKFold)"] = auc_cv(pipe_te(), dev[sin_cat + ["cliente"]], POR_CLIENTE, dev.cliente_id)

tabla = pd.Series(r, name="AUC").round(3)
display(tabla.to_frame())
fig, ax = plt.subplots(figsize=(8, 3.6))
colores = ["#2a9d8f" if "honesta" in k or "GroupKFold" in k else "#e76f51" for k in tabla.index]
ax.barh(tabla.index[::-1], tabla.values[::-1], color=colores[::-1])
ax.set_xlim(0.5, 1.02)
ax.set_xlabel("AUC (más alto parece mejor... ¡y ese es el peligro!)")
ax.set_title("Las fugas inflan la métrica sin mejorar el modelo")
plt.tight_layout()
fig.savefig(BASE / "fugas.png", dpi=120)
plt.show()

**Lectura de la tabla.** La referencia honesta (validación temporal, variables legítimas) es el nivel real. Todas las filas rojas lo superan *sin que el modelo sea mejor*: solo han dejado entrar información que en producción no existirá. Fíjate en dos detalles: el KFold aleatorio ya infla un poco el resultado con las mismas variables (mezcla pasado y futuro), y con `GroupKFold` el *target encoding* de cliente cae a su valor real.

**Mini-reto antes de seguir:** en la fuga 2, ¿qué ocurriría con `TimeSeriesSplit` si la tasa se calculara solo con el pasado? Es justo lo que hace `tasa_dev_prev`.

## 5 · Validación que imita el uso real

Dividimos **en el tiempo**, con huecos de un mes (el plazo de devolución es de 30 días, así que los pedidos recientes aún no tienen etiqueta fiable):

`entrenar → [hueco] → parada temprana → [hueco] → validación → [hueco] → prueba`

- `val` sirve para **elegir** (modelo, hiperparámetros, calibración, umbral). Se puede mirar muchas veces.
- `test` se mira **una vez**, al final. Si lo usas para decidir, deja de ser una estimación honesta.
- Dentro del entrenamiento, `TimeSeriesSplit` (con `gap`) para la validación cruzada; `GroupKFold` cuando no hay orden temporal pero sí clientes repetidos.

In [ ]:
# Partición temporal de TiendaSur (cada hueco de un mes deja cerrarse el plazo de devolución de 30 días)
f = df.fecha_pedido
MASCARAS = {
    "ent":  f < "2025-03-01",                              # entrenar
    "es":   (f >= "2025-04-01") & (f < "2025-06-01"),      # parada temprana (early stopping)
    "val":  (f >= "2025-07-01") & (f < "2025-10-01"),      # elegir modelo, afinar, calibrar y fijar umbral
    "test": f >= "2025-11-01",                             # solo se mira UNA vez, al final
}
for k, m in MASCARAS.items():
    print(f"{k:5s} {m.sum():6,d} pedidos  tasa de devolución {df.devuelto[m].mean():.1%}  "
          f"({f[m].min().date()} -> {f[m].max().date()})")
X_ent, y_ent = df.loc[MASCARAS["ent"], VARS], df.devuelto[MASCARAS["ent"]]
X_es, y_es = df.loc[MASCARAS["es"], VARS], df.devuelto[MASCARAS["es"]]
X_val, y_val = df.loc[MASCARAS["val"], VARS], df.devuelto[MASCARAS["val"]]
X_te, y_te = df.loc[MASCARAS["test"], VARS], df.devuelto[MASCARAS["test"]]

print("\nTimeSeriesSplit: cada pliegue entrena con el pasado y valida con el futuro inmediato")
for i, (a, b) in enumerate(TimeSeriesSplit(4, gap=1000).split(dev)):
    print(f"  pliegue {i}: entrena hasta {dev.fecha_pedido[a[-1]].date()}  ->  valida {dev.fecha_pedido[b[0]].date()} a {dev.fecha_pedido[b[-1]].date()}")

print("\nGroupKFold: ningún cliente aparece a la vez en entrenamiento y validación")
for a, b in GroupKFold(5).split(dev, groups=dev.cliente_id):
    assert not set(dev.cliente_id[a]) & set(dev.cliente_id[b])
print("  comprobado en los 5 pliegues")


def evaluar(y, p):
    return {"AUC": roc_auc_score(y, p), "AP": average_precision_score(y, p), "Brier": brier_score_loss(y, p)}


print(f"\nReferencia: predecir siempre la tasa base da AP = {y_val.mean():.3f} y AUC = 0.5. Todo se compara con eso.")

## 6 · XGBoost, LightGBM y HistGradientBoosting

Los tres hacen lo mismo (árboles pequeños que corrigen los errores de los anteriores, como en I3) con distinto motor:

| | Árboles | Categorías | Velocidad | Cuándo |
|---|---|---|---|---|
| `HistGradientBoostingClassifier` | por hojas | sí | media | ya tienes scikit-learn, sin dependencias |
| **LightGBM** | por hojas (*leaf-wise*) | sí | muy alta | tablas grandes, valor por defecto razonable |
| **XGBoost** | por niveles o por hojas | sí (`enable_categorical`) | alta | ecosistema enorme, GPU, producción conocida |

Hiperparámetros que de verdad importan: `learning_rate` (pequeño + muchos árboles), **número de árboles fijado por parada temprana**, tamaño de árbol (`num_leaves`/`max_depth`), `min_child_samples` (hoja mínima), regularización `reg_lambda` y submuestreo (`subsample`, `colsample_bytree`).

**Parada temprana:** el modelo vigila un conjunto aparte (`es`) y para cuando deja de mejorar. Sin ella, tienes que adivinar el nº de árboles; con ella, lo decide el dato. Compara también con la regresión logística de I3: es un baseline que hay que batir, no un trámite.

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler


def ajustar(motor, spw=1.0, **p):
    """Entrena un motor de boosting con parada temprana sobre el conjunto `es`. Devuelve (modelo, nº de árboles)."""
    lr, hojas, min_hoja, l2, muestreo = (p.get("lr", 0.05), p.get("hojas", 15), p.get("min_hoja", 40),
                                         p.get("l2", 5.0), p.get("muestreo", 0.8))
    n_max = p.get("n", 1000)
    if motor == "lightgbm":
        m = lgb.LGBMClassifier(n_estimators=n_max, learning_rate=lr, num_leaves=hojas, min_child_samples=min_hoja,
                               reg_lambda=l2, subsample=muestreo, subsample_freq=1, colsample_bytree=muestreo,
                               scale_pos_weight=spw, random_state=0, verbose=-1, n_jobs=4)
        m.fit(X_ent, y_ent, eval_set=[(X_es, y_es)], callbacks=[lgb.early_stopping(50, verbose=False)])
        return m, m.best_iteration_
    if motor == "xgboost":
        m = xgb.XGBClassifier(n_estimators=n_max, learning_rate=lr, max_leaves=hojas, grow_policy="lossguide",
                              min_child_weight=min_hoja / 10, reg_lambda=l2, subsample=muestreo,
                              colsample_bytree=muestreo, scale_pos_weight=spw, enable_categorical=True,
                              tree_method="hist", early_stopping_rounds=50, random_state=0, n_jobs=4)
        m.fit(X_ent, y_ent, eval_set=[(X_es, y_es)], verbose=False)
        return m, m.best_iteration + 1
    m = HistGradientBoostingClassifier(max_iter=n_max, learning_rate=lr, max_leaf_nodes=hojas, min_samples_leaf=min_hoja,
                                       l2_regularization=l2, categorical_features="from_dtype", early_stopping=True,
                                       n_iter_no_change=30, validation_fraction=0.15, random_state=0)
    peso = np.where(y_ent == 1, spw, 1.0)
    m.fit(X_ent, y_ent, sample_weight=peso)
    return m, m.n_iter_


# Referencia lineal (módulo I3): regresión logística con preprocesado
num = [c for c in VARS if c not in VARS_CAT]
logit = make_pipeline(
    ColumnTransformer([("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num),
                       ("cat", OneHotEncoder(handle_unknown="ignore"), VARS_CAT)]),
    LogisticRegression(max_iter=1000))

resultados = {}
t0 = time.time(); logit.fit(X_ent, y_ent)
resultados["Regresión logística (I3)"] = {**evaluar(y_val, logit.predict_proba(X_val)[:, 1]), "árboles": "-", "seg": time.time() - t0}

modelos = {}
for motor in [m for m, ok in [("hgb", True), ("xgboost", xgb), ("lightgbm", lgb)] if ok]:
    t0 = time.time()
    modelo, n_arboles = ajustar(motor)
    modelos[motor] = modelo
    resultados[{"hgb": "HistGradientBoosting", "xgboost": "XGBoost", "lightgbm": "LightGBM"}[motor]] = {
        **evaluar(y_val, modelo.predict_proba(X_val)[:, 1]), "árboles": n_arboles, "seg": time.time() - t0}

tabla_modelos = pd.DataFrame(resultados).T
display(tabla_modelos.round(3))
print(f"Tasa base en validación: {y_val.mean():.3f}  (AP de un modelo inútil)")
print("Con parada temprana, cada motor elige solo cuántos árboles necesita; las diferencias entre ellos son pequeñas.")

## 7 · Clases desequilibradas: ¿ponderar o no?

Las devoluciones son una minoría. La reacción habitual es `scale_pos_weight` o `class_weight="balanced"`. Probemos si ayuda y qué cuesta.

In [ ]:
# Desequilibrio: ~14 % de devoluciones. ¿Ayuda ponderar la clase positiva?
neg, pos = (y_ent == 0).sum(), (y_ent == 1).sum()
filas = {}
for nombre, spw in [("sin ponderar (1)", 1.0), ("raíz del cociente", float(np.sqrt(neg / pos))), ("cociente completo", float(neg / pos))]:
    modelo, n = ajustar(MOTOR, spw=spw)
    p = modelo.predict_proba(X_val)[:, 1]
    filas[nombre] = {"scale_pos_weight": round(spw, 2), **evaluar(y_val, p), "prob. media predicha": p.mean(), "tasa real": y_val.mean()}
display(pd.DataFrame(filas).T.round(3))
print("Ponderar NO mejoró el orden de los pedidos (AUC/AP iguales o peores) y sí estropeó las probabilidades: la media")
print("predicha se aleja de la tasa real y el Brier empeora. Con un 12 % de positivos no hay un desequilibrio que justifique")
print("ponderar. Si necesitas probabilidades (y para decidir por coste las necesitas), no ponderes, o calibra después.")

## 8 · Búsqueda de hiperparámetros sensata

Con presupuesto pequeño (8 pruebas) comparamos dos estrategias sobre **el mismo objetivo**: AP medio en validación cruzada temporal sobre el pasado (nunca tocamos `val` ni `test`).
- **Aleatoria:** sorprendentemente fuerte; mejor que una rejilla porque explora más valores de lo importante.
- **Optuna (TPE):** aprende de las pruebas anteriores para proponer las siguientes. Gana cuando hay más presupuesto.

Idea clave: la búsqueda tiene retornos decrecientes. Más datos y mejores variables pesan más que 200 pruebas.

In [ ]:
# Búsqueda de hiperparámetros: mismo objetivo, dos formas de explorar (aleatoria vs Optuna/TPE)
ESPACIO = {"lr": (0.02, 0.2, "log"), "hojas": (4, 48, "int"), "min_hoja": (10, 200, "int"),
           "l2": (1.0, 50.0, "log"), "n": (100, 300, "int"), "muestreo": (0.6, 1.0, "float")}
X_dev_ent = pd.concat([X_ent, X_es]); y_dev_ent = pd.concat([y_ent, y_es])        # el pasado hasta junio
CV_TIEMPO = TimeSeriesSplit(3, gap=1000)


def fabricar(motor, **p):
    """Modelo SIN parada temprana (n fijo) para usar dentro de la validación cruzada."""
    if motor == "lightgbm":
        return lgb.LGBMClassifier(n_estimators=p["n"], learning_rate=p["lr"], num_leaves=p["hojas"], min_child_samples=p["min_hoja"],
                                  reg_lambda=p["l2"], subsample=p["muestreo"], subsample_freq=1, colsample_bytree=p["muestreo"],
                                  random_state=0, verbose=-1, n_jobs=4)
    if motor == "xgboost":
        return xgb.XGBClassifier(n_estimators=p["n"], learning_rate=p["lr"], max_leaves=p["hojas"], grow_policy="lossguide",
                                 min_child_weight=p["min_hoja"] / 10, reg_lambda=p["l2"], subsample=p["muestreo"],
                                 colsample_bytree=p["muestreo"], enable_categorical=True, tree_method="hist", random_state=0, n_jobs=4)
    return HistGradientBoostingClassifier(max_iter=p["n"], learning_rate=p["lr"], max_leaf_nodes=p["hojas"], min_samples_leaf=p["min_hoja"],
                                          l2_regularization=p["l2"], categorical_features="from_dtype", random_state=0)


def objetivo(p):
    """AP medio en validación temporal: la métrica que de verdad nos importa, sin tocar `val` ni `test`."""
    return cross_val_score(fabricar(MOTOR, **p), X_dev_ent, y_dev_ent, cv=CV_TIEMPO, scoring="average_precision").mean()


def sugerir(trial):
    p = {}
    for k, (a, b, tipo) in ESPACIO.items():
        p[k] = trial.suggest_int(k, a, b) if tipo == "int" else trial.suggest_float(k, a, b, log=(tipo == "log"))
    return p


N_PRUEBAS = 8
t0 = time.time()
por_defecto = {"lr": 0.05, "hojas": 15, "min_hoja": 40, "l2": 5.0, "n": 200, "muestreo": 0.8}
print(f"AP (CV temporal) con valores por defecto: {objetivo(por_defecto):.4f}")

if optuna:
    estudios = {}
    for nombre, sampler in [("aleatoria", optuna.samplers.RandomSampler(seed=0)), ("Optuna TPE", optuna.samplers.TPESampler(seed=0))]:
        estudio = optuna.create_study(direction="maximize", sampler=sampler)
        estudio.optimize(lambda t: objetivo(sugerir(t)), n_trials=N_PRUEBAS)
        estudios[nombre] = estudio
        print(f"{nombre:11s} mejor AP (CV temporal) tras {N_PRUEBAS} pruebas: {estudio.best_value:.4f}")
    mejor = max(estudios.values(), key=lambda e: e.best_value).best_params
else:
    rng = np.random.default_rng(0)
    intentos = []
    for _ in range(N_PRUEBAS):
        p = {k: (int(rng.integers(a, b + 1)) if t == "int" else float(np.exp(rng.uniform(np.log(a), np.log(b)))) if t == "log" else float(rng.uniform(a, b)))
             for k, (a, b, t) in ESPACIO.items()}
        intentos.append((objetivo(p), p))
    print(f"aleatoria   mejor AP (CV temporal) tras {N_PRUEBAS} pruebas: {max(intentos, key=lambda x: x[0])[0]:.4f}")
    mejor = max(intentos, key=lambda x: x[0])[1]
print(f"Búsqueda terminada en {time.time() - t0:.0f} s. Mejores hiperparámetros:", {k: round(v, 3) if isinstance(v, float) else v for k, v in mejor.items()})

# Modelo final con esos hiperparámetros y parada temprana (n se vuelve a decidir con `es`)
mejor_sin_n = {k: v for k, v in mejor.items() if k != "n"}
modelo_base, n_base = ajustar(MOTOR)
modelo_afinado, n_afinado = ajustar(MOTOR, **mejor_sin_n)
for nombre, m, n in [("por defecto", modelo_base, n_base), ("afinado", modelo_afinado, n_afinado)]:
    print(f"  {nombre:11s} ({n} árboles) -> validación: " + ", ".join(f"{k} {v:.4f}" for k, v in evaluar(y_val, m.predict_proba(X_val)[:, 1]).items()))
print("Si la mejora es pequeña, es normal: el valor está en los datos y las variables, no en el último decimal.")
modelo_final = modelo_afinado if evaluar(y_val, modelo_afinado.predict_proba(X_val)[:, 1])["AP"] >= evaluar(y_val, modelo_base.predict_proba(X_val)[:, 1])["AP"] else modelo_base
p_val = modelo_final.predict_proba(X_val)[:, 1]
p_te = modelo_final.predict_proba(X_te)[:, 1]
print("Modelo final elegido con la validación. En prueba (primera y última vez que se mira): "
      + ", ".join(f"{k} {v:.4f}" for k, v in evaluar(y_te, p_te).items()))

## 9 · Interpretabilidad: permutación y SHAP

- **Importancia por permutación:** baraja una columna y mide cuánto cae la métrica. Funciona con cualquier modelo y se mide sobre datos no vistos.
- **SHAP:** reparte la predicción de **cada pedido** entre las variables (valores de Shapley). Sumando sobre pedidos obtienes importancia global; mirando uno, explicación local. Para árboles es rápido y exacto (`TreeExplainer`).

Qué mirar en el gráfico de abejas (*beeswarm*): el orden vertical es la importancia; el color, el valor de la variable; la posición horizontal, si empuja el riesgo arriba (derecha) o abajo (izquierda).

In [ ]:
from sklearn.inspection import permutation_importance

# 1) Importancia por permutación (agnóstica del modelo): ¿cuánto empeora AP al barajar una variable?
t0 = time.time()
perm = permutation_importance(modelo_final, X_val, y_val, scoring="average_precision", n_repeats=5, random_state=0, n_jobs=1)
imp_perm = pd.Series(perm.importances_mean, index=VARS).sort_values(ascending=False)
print(f"Importancia por permutación (caída de AP en validación; {time.time() - t0:.0f} s):")
display(imp_perm.head(8).round(4).to_frame("caída de AP"))

# 2) SHAP (específico de árboles): cuánto suma o resta cada variable a la predicción de CADA pedido
if shap and MOTOR != "hgb":
    muestra = X_val.sample(1500, random_state=0)
    explicador = shap.TreeExplainer(modelo_final)
    sv = explicador(muestra)
    if sv.values.ndim == 3:            # algunas versiones devuelven una capa por clase
        sv = sv[:, :, 1]
    print("SHAP calculado para", len(muestra), "pedidos;  valores en escala log-odds, forma", sv.values.shape)

    plt.figure(); shap.plots.beeswarm(sv, max_display=12, show=False)
    plt.title("SHAP global: qué variables mueven el riesgo y en qué dirección"); plt.tight_layout()
    plt.savefig(BASE / "shap_resumen.png", dpi=120, bbox_inches="tight"); plt.show()

    plt.figure(); shap.plots.bar(sv, max_display=12, show=False)
    plt.tight_layout(); plt.savefig(BASE / "shap_barras.png", dpi=120, bbox_inches="tight"); plt.show()

    imp_shap = pd.Series(np.abs(sv.values).mean(0), index=muestra.columns).sort_values(ascending=False)
    comparacion = pd.DataFrame({"rango SHAP": imp_shap.rank(ascending=False), "rango permutación": imp_perm.rank(ascending=False)}).sort_values("rango SHAP")
    display(comparacion.head(8))
    print("Dos métodos, dos definiciones de «importante»: si coinciden en lo esencial, la conclusión es robusta.")
else:
    print("SHAP necesita LightGBM o XGBoost operativos; aquí seguimos solo con la importancia por permutación.")

### Dependencia y explicación local

El gráfico de dependencia muestra cómo cambia la contribución de una variable con su valor (y los puntos dispersos verticalmente delatan interacciones). La cascada (*waterfall*) explica un único pedido: parte del riesgo medio y suma/resta cada variable.

In [ ]:
if shap and MOTOR != "hgb":
    # Dependencia: cómo cambia la contribución de una variable según su valor
    fig, ejes = plt.subplots(1, 2, figsize=(11, 3.8))
    shap.plots.scatter(sv[:, "descuento_pct"], ax=ejes[0], show=False)
    shap.plots.scatter(sv[:, "tasa_dev_prev"], ax=ejes[1], show=False)
    plt.tight_layout(); fig.savefig(BASE / "shap_dependencia.png", dpi=120, bbox_inches="tight"); plt.show()

    # Explicación local: un pedido concreto (el de mayor riesgo de la muestra)
    i = int(np.argmax(sv.base_values + sv.values.sum(1)))
    plt.figure(); shap.plots.waterfall(sv[i], max_display=9, show=False)
    plt.tight_layout(); plt.savefig(BASE / "shap_local.png", dpi=120, bbox_inches="tight"); plt.show()
    print("Pedido explicado:\n", muestra.iloc[i][["categoria", "canal", "importe", "descuento_pct", "tasa_dev_prev", "dias_entrega"]].to_dict())
    print("\nAdvertencia: SHAP explica el MODELO, no el MUNDO. «El descuento sube el riesgo» significa que el modelo usa el")
    print("descuento para predecir devoluciones; no prueba que quitar descuentos reduzca devoluciones (eso requiere un")
    print("experimento A/B; ver módulos de causalidad). Variables correlacionadas se reparten el mérito de forma arbitraria.")

## 10 · Calibración de probabilidades

Un modelo puede **ordenar bien** (AUC alto) y aun así dar probabilidades que no son probabilidades: si dice «30 %» a mil pedidos, ¿se devuelven ~300? La **curva de fiabilidad** lo comprueba. Para decidir por coste necesitamos probabilidades fiables, porque comparamos `p × beneficio` con un coste en euros.

Calibramos con un conjunto que el modelo no usó para entrenar (`val`) y medimos en `test`. Dos métodos: **sigmoide** (Platt; pocos datos) e **isotónica** (más flexible; necesita más datos). `FrozenEstimator` indica que el modelo ya está entrenado y solo se ajusta el calibrador.

In [ ]:
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.frozen import FrozenEstimator


def ece(y, p, bins=10):
    """Error de calibración esperado (ECE): diferencia media entre probabilidad predicha y frecuencia real."""
    cortes = np.quantile(p, np.linspace(0, 1, bins + 1)); cortes[-1] += 1e-9
    idx = np.digitize(p, cortes[1:-1])
    return float(sum((idx == b).mean() * abs(p[idx == b].mean() - np.asarray(y)[idx == b].mean()) for b in range(bins) if (idx == b).any()))


# Modelo ponderado: buen ORDEN, probabilidades infladas -> lo calibramos con la validación y medimos en prueba
modelo_pond, _ = ajustar(MOTOR, spw=float(neg / pos))
p_pond = modelo_pond.predict_proba(X_te)[:, 1]
sigmoide = CalibratedClassifierCV(FrozenEstimator(modelo_pond), method="sigmoid").fit(X_val, y_val)
isotonica = CalibratedClassifierCV(FrozenEstimator(modelo_pond), method="isotonic").fit(X_val, y_val)

curvas = {"Ponderado (sin calibrar)": p_pond,
          "Ponderado + sigmoide": sigmoide.predict_proba(X_te)[:, 1],
          "Ponderado + isotónica": isotonica.predict_proba(X_te)[:, 1],
          "Modelo final (sin ponderar)": p_te}
fig, ejes = plt.subplots(1, 2, figsize=(11, 4))
ejes[0].plot([0, 0.6], [0, 0.6], "k--", label="perfecta")
filas = {}
for nombre, p in curvas.items():
    frec, med = calibration_curve(y_te, p, n_bins=8, strategy="quantile")
    ejes[0].plot(med, frec, "o-", label=nombre)
    filas[nombre] = {"Brier": brier_score_loss(y_te, p), "ECE": ece(y_te, p), "AUC": roc_auc_score(y_te, p), "prob. media": p.mean()}
ejes[0].set_xlabel("probabilidad predicha"); ejes[0].set_ylabel("frecuencia real de devolución")
ejes[0].set_title("Curva de fiabilidad (prueba)"); ejes[0].legend(fontsize=8)
ejes[1].hist(p_te, bins=40, color="#264653"); ejes[1].set_title("Distribución de probabilidades del modelo final")
plt.tight_layout(); fig.savefig(BASE / "calibracion.png", dpi=120); plt.show()
display(pd.DataFrame(filas).T.round(4))
print(f"Tasa real en prueba: {y_te.mean():.3f}.  La calibración mejora Brier/ECE y deja el AUC casi intacto: cambia los valores, no el orden.")

## 11 · El umbral lo decide el negocio

Enlaza con la lección 4.6 de I3. Una acción preventiva cuesta `c`; si el pedido iba a devolverse y la acción funciona (eficacia `e`), ahorramos `e × D` (D = coste de una devolución). Actuar compensa cuando `p × e × D > c`, es decir `p > c / (e × D)`. Ese es el **umbral teórico**; el empírico se busca en `val` barriendo umbrales y se confirma en `test`.

In [ ]:
# Umbral por coste de negocio (módulo I3, lección 4.6). Supuestos de TiendaSur (ajústalos con Finanzas):
COSTE_DEV = 18.0      # € que cuesta una devolución: transporte inverso, revisión, reempaquetado, merma
COSTE_ACCION = 1.2    # € por pedido marcado: verificación de talla, mensaje proactivo, envío con comprobación
EFICACIA = 0.40       # fracción de devoluciones que la acción evita


def beneficio(y, p, umbral):
    marcado = np.asarray(p) >= umbral
    return float((marcado * (np.asarray(y) * EFICACIA * COSTE_DEV - COSTE_ACCION)).sum())


umbral_teorico = COSTE_ACCION / (EFICACIA * COSTE_DEV)
print(f"Umbral teórico (coste de actuar = beneficio esperado): p* = {COSTE_ACCION}/({EFICACIA}*{COSTE_DEV}) = {umbral_teorico:.3f}")

# Calibramos el modelo final con la validación (así las probabilidades son interpretables como probabilidades)
calibrado = CalibratedClassifierCV(FrozenEstimator(modelo_final), method="isotonic").fit(X_val, y_val)
p_val_c, p_te_c = calibrado.predict_proba(X_val)[:, 1], calibrado.predict_proba(X_te)[:, 1]

umbrales = np.linspace(0.02, 0.8, 79)
b_val = np.array([beneficio(y_val, p_val_c, u) for u in umbrales])
u_opt = float(umbrales[b_val.argmax()])
print(f"Umbral óptimo empírico en VALIDACIÓN: {u_opt:.2f}")

n_te = len(y_te)
politicas = {
    "No hacer nada": 0.0,
    "Marcar todos los pedidos": beneficio(y_te, p_te_c, 0.0),
    "Umbral 0,5 (el «por defecto»)": beneficio(y_te, p_te_c, 0.5),
    f"Umbral teórico {umbral_teorico:.2f}": beneficio(y_te, p_te_c, umbral_teorico),
    f"Umbral óptimo en validación {u_opt:.2f}": beneficio(y_te, p_te_c, u_opt),
}
tabla_umbral = pd.DataFrame({"beneficio en prueba (€)": politicas}).assign(
    **{"€ por 1.000 pedidos": lambda t: t.iloc[:, 0] / n_te * 1000,
       "pedidos marcados": [0, n_te] + [int((p_te_c >= u).sum()) for u in (0.5, umbral_teorico, u_opt)]})
display(tabla_umbral.round(1))

fig, ax = plt.subplots(figsize=(7.5, 3.6))
ax.plot(umbrales, b_val / len(y_val) * 1000, label="validación")
ax.plot(umbrales, [beneficio(y_te, p_te_c, u) / n_te * 1000 for u in umbrales], label="prueba")
ax.axvline(u_opt, color="gray", ls=":"); ax.axhline(0, color="k", lw=0.6)
ax.set_xlabel("umbral de decisión"); ax.set_ylabel("€ por cada 1.000 pedidos"); ax.legend(); ax.set_title("Beneficio según el umbral")
plt.tight_layout(); fig.savefig(BASE / "umbral.png", dpi=120); plt.show()
print("El umbral lo fija el negocio (costes y capacidad), no el 0,5 de la costumbre. Y se elige en validación, no en prueba.")

## 12 · El informe de interpretabilidad (la pieza de portafolio)

Todo lo anterior acaba en un documento que entiende una persona de negocio: qué predice el modelo, cuánto acierta, **qué empuja el riesgo**, qué hacer, cuánto dinero está en juego y qué límites tiene. Aquí se genera automáticamente en `proyecto-devoluciones/informe_interpretabilidad.md`; en tu caso práctico lo redactarás tú con tus datos.

In [ ]:
import json
from scipy.stats import spearmanr

m_te = evaluar(y_te, p_te)
mejor_b = politicas[f"Umbral óptimo en validación {u_opt:.2f}"]
lineas_drivers = []
if shap and MOTOR != "hgb":
    top = pd.Series(np.abs(sv.values).mean(0), index=muestra.columns).sort_values(ascending=False).head(5)
    for var in top.index:
        if str(muestra[var].dtype) == "category":
            sentido = "según el valor de la categoría"
        else:
            rho = spearmanr(muestra[var].fillna(muestra[var].median()), sv[:, var].values)[0]
            sentido = "el riesgo SUBE cuando la variable aumenta" if rho > 0.15 else "el riesgo BAJA cuando la variable aumenta" if rho < -0.15 else "efecto no monótono"
        lineas_drivers.append(f"| `{var}` | {top[var]:.3f} | {sentido} |")
else:
    for var in imp_perm.head(5).index:
        lineas_drivers.append(f"| `{var}` | {imp_perm[var]:.4f} (permutación) | ver gráficos |")

imagen_shap = "![SHAP](shap_resumen.png)" if (shap and MOTOR != "hgb") else ""
informe = f"""# Informe de interpretabilidad · Modelo de devoluciones de TiendaSur

*Generado automáticamente por el notebook del módulo I4. Datos simulados.*

## 1. Resumen para dirección
- El modelo estima, en el momento de la compra, la probabilidad de que un pedido se devuelva.
- Sobre pedidos que no vio al entrenar (noviembre-diciembre de 2025) distingue las devoluciones con un AUC de **{m_te['AUC']:.2f}** y una precisión media (AP) de **{m_te['AP']:.2f}**, frente a **{y_te.mean():.2f}** de un modelo sin información.
- Con los costes supuestos (devolución {COSTE_DEV:.0f} €, acción preventiva {COSTE_ACCION:.1f} €, eficacia {EFICACIA:.0%}), actuar sobre los pedidos con riesgo ≥ **{u_opt:.2f}** habría aportado unos **{mejor_b:,.0f} €** en esos dos meses ({mejor_b / n_te * 1000:,.0f} € por cada 1.000 pedidos).

## 2. Qué empuja el riesgo de devolución
| Variable | Peso medio (|SHAP|) | Lectura |
|---|---|---|
{chr(10).join(lineas_drivers)}

{imagen_shap}

## 3. Cómo usar el modelo
1. Calcular el riesgo al registrar el pedido.
2. Si el riesgo supera {u_opt:.2f}: lanzar la acción preventiva (verificación de talla, mensaje, envío con comprobación).
3. Revisar el umbral cada trimestre: depende de los costes y de la capacidad del equipo, no del modelo.

## 4. Limitaciones y advertencias
- **Correlación no es causalidad.** Las explicaciones describen qué usa el modelo, no qué *causa* las devoluciones. Antes de, por ejemplo, retirar descuentos, hay que hacer un experimento A/B.
- **Datos simulados:** los números concretos no son los de una tienda real; sí lo es el método.
- **Deriva:** en abril de 2025 hay un lote defectuoso de máquinas. Modelos así envejecen: hay que monitorizar (módulo I8) y reentrenar.
- **Equidad:** comprobar que la acción preventiva no perjudica de forma sistemática a ciertos clientes.
- La estimación de beneficio depende de supuestos de coste (ver código); con otros costes cambia el umbral.

## 5. Archivos
`fugas.png`, `shap_resumen.png`, `shap_barras.png`, `shap_dependencia.png`, `shap_local.png`, `calibracion.png`, `umbral.png`, `ficha_modelo.json`
"""
(BASE / "informe_interpretabilidad.md").write_text(informe)
ficha = {"motor": MOTOR, "variables": VARS, "umbral": u_opt, "metricas_prueba": {k: round(v, 4) for k, v in m_te.items()},
         "datos": "simulados (TiendaSur)", "entrenamiento": "pedidos anteriores a 2025-03-01", "uso_previsto": "priorizar acciones preventivas, no denegar devoluciones"}
(BASE / "ficha_modelo.json").write_text(json.dumps(ficha, indent=2, ensure_ascii=False))
print(informe[:1800])
print("... informe completo en", BASE / "informe_interpretabilidad.md")

## Ejercicios

Resuélvelos en las celdas vacías de abajo; puedes reutilizar cualquier variable del notebook (`df`, `VARS`, `ajustar`, `X_val`, `p_val_c`, etc.).

**Ejercicio 1 · Una variable de historial nueva.** Crea `pct_desc_prev` (proporción de pedidos **anteriores** del cliente que tuvieron descuento) y `es_primer_pedido`. Añádelas a `VARS`, entrena un `HistGradientBoostingClassifier` con y sin ellas sobre `ent` y compara el **AP en `val`**. Vigila que no uses el pedido actual ni el futuro.

**Ejercicio 2 · Auditoría de fugas.** Construye este DataFrame y calcula, para cada columna sospechosa, el **AUC de usarla sola** como puntuación en `val`. Marca cuáles te parecen fugas y justifícalo en una frase cada una:
```python
sospechoso = df.loc[MASCARAS["val"]].copy()
sospechoso["importe_reembolsado"] = sospechoso.importe * sospechoso.devuelto
sospechoso["n_pedidos_cliente_total"] = df.groupby("cliente_id").pedido_id.transform("size")[sospechoso.index]
sospechoso["dias_entrega_real"] = sospechoso.dias_entrega
```

**Ejercicio 3 · Desequilibrio y calibración.** Entrena con `scale_pos_weight` ∈ {1, 3} (con `ajustar`), mide **ECE y Brier en `test`**, y calibra con sigmoide usando `val`. ¿Qué configuración queda mejor calibrada y por qué?

**Ejercicio 4 · Otros costes.** Una devolución cuesta 30 €, la acción cuesta 5 € y funciona el 25 % de las veces. Calcula el umbral teórico, el empírico en `val` y el beneficio en `test`. ¿Compensa actuar?

**Ejercicio 5 · Explicar un pedido.** Con SHAP, toma el pedido de mayor riesgo de `test` y escribe en 3 frases lo que le dirías a una persona de atención al cliente, indicando las tres variables que más pesan. Incluye una advertencia sobre causalidad.

In [ ]:
# Ejercicio 1 · tu código aquí


In [ ]:
# Ejercicio 2 · tu código aquí


In [ ]:
# Ejercicio 3 · tu código aquí


In [ ]:
# Ejercicio 4 · tu código aquí


In [ ]:
# Ejercicio 5 · tu código aquí
